# Atelier Préparation de Données Images 

## Partie 1 – Exploration du dataset 

Développer un programme Python capable de récupérer, pour chaque image, son nom, sa classe, son format, son mode, sa largeur, sa hauteur, l’écart-type de ses pixels, son nombre de canaux et sa taille. 

NB : prendre en charge aussi les fichiers corrompus 

### Cellule de code 1 imports

In [1]:
import os                      # pour parcourir les dossiers et récupérer la taille des fichiers
import numpy as np             # pour calculer l'écart-type des pixels
from PIL import Image          # pour ouvrir et lire les métadonnées des images
import pandas as pd            # pour stocker les résultats dans un tableau (DataFrame)

RAW_DIR = "../data/raw"        # chemin vers le dossier contenant les images brutes, par classe

### Cellule de code 2 fonction d'extraction pour une image

In [2]:
def extraire_infos_image(chemin_fichier, classe):
    # chemin_fichier : chemin complet vers l'image à analyser
    # classe : nom du sous-dossier (cardboard, glass, metal, paper, plastic, trash)

    infos = {
        "nom": os.path.basename(chemin_fichier),   # nom du fichier seul (sans le chemin)
        "classe": classe,                          # classe déduite du dossier parent
        "format": None,                            # sera rempli si l'image s'ouvre correctement
        "mode": None,
        "largeur": None,
        "hauteur": None,
        "ecart_type_pixels": None,
        "nb_canaux": None,
        "taille_octets": os.path.getsize(chemin_fichier),  # taille du fichier sur le disque
        "corrompue": False,                         # drapeau : True si l'ouverture échoue
    }

    try:
        with Image.open(chemin_fichier) as img:     # tentative d'ouverture de l'image
            img.verify()                            # vérifie l'intégrité du fichier (détecte la corruption)
        # img.verify() rend l'objet inutilisable pour la suite, donc on rouvre l'image
        with Image.open(chemin_fichier) as img:
            infos["format"] = img.format            # ex: JPEG, PNG
            infos["mode"] = img.mode                 # ex: RGB, L (grayscale), RGBA
            infos["largeur"], infos["hauteur"] = img.size  # (largeur, hauteur) en pixels

            tableau_pixels = np.array(img)           # conversion de l'image en tableau numpy
            infos["ecart_type_pixels"] = tableau_pixels.std()  # écart-type, tous canaux aplatis automatiquement

            # nombre de canaux : 1 si l'image est 2D (grayscale), sinon la 3e dimension du tableau
            infos["nb_canaux"] = 1 if tableau_pixels.ndim == 2 else tableau_pixels.shape[2]

    except Exception:
        # si l'ouverture ou la lecture échoue, on marque l'image comme corrompue
        # et on laisse les autres champs à None plutôt que de faire planter le programme
        infos["corrompue"] = True

    return infos

### Cellule de code 3 parcours du dataset et construction du DataFrame

In [3]:
EXTENSIONS_VALIDES = (".jpg", ".jpeg", ".png")  # extensions d'images qu'on accepte

resultats = []  # liste qui va accueillir un dictionnaire d'infos par image

# on récupère la liste des classes = noms des sous-dossiers de RAW_DIR
classes = sorted(os.listdir(RAW_DIR))

for classe in classes:
    dossier_classe = os.path.join(RAW_DIR, classe)   # ex: ../data/raw/cardboard

    if not os.path.isdir(dossier_classe):
        continue  # on ignore les éléments qui ne sont pas des dossiers

    for nom_fichier in os.listdir(dossier_classe):
        chemin_fichier = os.path.join(dossier_classe, nom_fichier)  # chemin complet vers le fichier

        if not os.path.isfile(chemin_fichier):
            continue  # on ignore les sous-dossiers éventuels

        if not nom_fichier.lower().endswith(EXTENSIONS_VALIDES):
            continue  # on ignore les fichiers qui ne sont pas des images (ex: Zone.Identifier)

        infos = extraire_infos_image(chemin_fichier, classe)  # appel de la fonction de la cellule 2
        resultats.append(infos)  # ajout du dictionnaire à la liste globale

# conversion de la liste de dictionnaires en tableau pandas : une ligne = une image
df_audit = pd.DataFrame(resultats)

print(f"Nombre total d'images analysées : {len(df_audit)}")
df_audit.head()  # aperçu des premières lignes du tableau

Nombre total d'images analysées : 1030


,nom,classe,format,mode,largeur,hauteur,ecart_type_pixels,nb_canaux,taille_octets,corrompue
0,cardboard95.jpg,cardboard,JPEG,RGB,512.0,384.0,75.067740,3.0,30705,False
1,cardboard51.jpg,cardboard,JPEG,RGB,512.0,384.0,45.727011,3.0,21683,False
2,cardboard111.jpg,cardboard,JPEG,RGB,512.0,384.0,45.872520,3.0,24881,False
3,cardboard24.jpg,cardboard,JPEG,RGB,512.0,384.0,42.975044,3.0,19051,False
4,cardboard98.jpg,cardboard,JPEG,RGB,512.0,384.0,56.502087,3.0,21538,False


In [4]:
df_audit["classe"].value_counts()

classe
paper        252
plastic      224
glass        187
cardboard    168
metal        149
trash         50
Name: count, dtype: int64

## Partie 2 – Détecter les images corrompues

Écrire et se servir d’une fonction qui détecte une image corrompue. 

In [5]:
def est_corrompue(chemin_fichier):
    # tente d'ouvrir et de vérifier l'intégrité du fichier
    # renvoie True si le fichier est corrompu, False s'il est valide
    try:
        with Image.open(chemin_fichier) as img:
            img.verify()
        return False
    except Exception:
        return True


# on reparcourt le dataset en appliquant uniquement cette fonction,
# en réutilisant le même filtre d'extension que la Partie 1
images_corrompues = []  # chemins des images détectées comme corrompues

for classe in classes:  # variable déjà définie à la Partie 1
    dossier_classe = os.path.join(RAW_DIR, classe)

    if not os.path.isdir(dossier_classe):
        continue

    for nom_fichier in os.listdir(dossier_classe):
        chemin_fichier = os.path.join(dossier_classe, nom_fichier)

        if not os.path.isfile(chemin_fichier):
            continue

        if not nom_fichier.lower().endswith(EXTENSIONS_VALIDES):
            continue  # on ignore les fichiers non-images (ex: Zone.Identifier)

        if est_corrompue(chemin_fichier):
            images_corrompues.append(chemin_fichier)

print(f"Nombre d'images corrompues détectées : {len(images_corrompues)}")
images_corrompues

Nombre d'images corrompues détectées : 6


['../data/raw/cardboard/cardboard83.jpg',
 '../data/raw/glass/glass74.jpg',
 '../data/raw/metal/metal48.jpg',
 '../data/raw/paper/paper213.jpg',
 '../data/raw/plastic/plastic13.jpg',
 '../data/raw/trash/trash3.jpg']

## Partie 3 – Détecter les images vides

Écrire et se servir d’une fonction qui détecte les images vides : image entièrement noire, image entièrement blanche ou image dont les pixels présentent très peu de variation. 

In [6]:
df_audit["ecart_type_pixels"].describe()

count    1024.000000
mean       50.315155
std        15.932928
min         1.572536
25%        38.829644
50%        49.946351
75%        61.867152
max       110.418239
Name: ecart_type_pixels, dtype: float64

In [7]:
df_audit["ecart_type_pixels"].sort_values().head(15)

65      1.572536
397     1.572536
948     9.597787
825     9.727732
975    10.769687
945    10.823302
914    11.224502
219    12.546322
176    13.001910
184    14.113714
346    16.160845
300    16.940087
169    17.208479
171    17.324620
182    18.495063
Name: ecart_type_pixels, dtype: float64

In [8]:
SEUIL_ECART_TYPE_VIDE = 5  # seuil déterminé après analyse de la distribution (saut net à 1.57 vs 9.6+)

df_audit["vide"] = df_audit["ecart_type_pixels"] < SEUIL_ECART_TYPE_VIDE

images_vides = df_audit[df_audit["vide"] == True]

print(f"Nombre d'images vides détectées : {len(images_vides)}")
images_vides[["nom", "classe", "ecart_type_pixels"]]

Nombre d'images vides détectées : 2


,nom,classe,ecart_type_pixels
65,image-blanche-512x384.jpg,cardboard,1.572536
397,image-blanche-512x384.jpg,metal,1.572536


## Partie 4 – Détecter les différences de résolution

### 1) Déterminer la résolution minimale, la résolution maximale, les résolutions les plus fréquentes et le nombre d'images par résolution. 

In [9]:
# on ne travaille que sur les images valides (non corrompues), car largeur/hauteur sont NaN sinon
df_valides = df_audit[df_audit["corrompue"] == False].copy()

# on combine largeur et hauteur en une chaîne "LxH" pour identifier chaque résolution distincte
df_valides["resolution"] = (
    df_valides["largeur"].astype(int).astype(str)
    + "x"
    + df_valides["hauteur"].astype(int).astype(str)
)

# nombre d'images par résolution, trié du plus fréquent au moins fréquent
resolutions_frequentes = df_valides["resolution"].value_counts()
print("Résolutions les plus fréquentes :")
print(resolutions_frequentes.head(10))

# calcul de la surface (largeur x hauteur) pour chaque image
df_valides["surface"] = df_valides["largeur"] * df_valides["hauteur"]

# image avec la plus petite surface
image_min = df_valides.loc[df_valides["surface"].idxmin()]
print(f"\nImage la plus petite : {image_min['nom']} ({image_min['resolution']})")

# image avec la plus grande surface
image_max = df_valides.loc[df_valides["surface"].idxmax()]
print(f"Image la plus grande : {image_max['nom']} ({image_max['resolution']})")

Résolutions les plus fréquentes :
resolution
512x384    1011
32x32         5
40x40         4
48x32         4
Name: count, dtype: int64

Image la plus petite : cardboard22.jpg (32x32)
Image la plus grande : cardboard95.jpg (512x384)


### 2) On décide qu'une image doit avoir au minimum 64 × 64 pixels. Identifier toutes les images ne respectant pas cette contrainte. 

In [10]:
SEUIL_MIN_PIXELS = 64  # taille minimale acceptée dans chaque dimension

images_trop_petites = df_valides[
    (df_valides["largeur"] < SEUIL_MIN_PIXELS) | (df_valides["hauteur"] < SEUIL_MIN_PIXELS)
]

print(f"Nombre d'images trop petites (< {SEUIL_MIN_PIXELS}x{SEUIL_MIN_PIXELS}) : {len(images_trop_petites)}")
images_trop_petites[["nom", "classe", "resolution"]]

Nombre d'images trop petites (< 64x64) : 13


,nom,classe,resolution
21,cardboard70.jpg,cardboard,40x40
84,cardboard22.jpg,cardboard,32x32
107,cardboard117.jpg,cardboard,48x32
171,glass15.jpg,glass,48x32
209,glass23.jpg,glass,32x32
336,glass100.jpg,glass,40x40
342,glass21.jpg,glass,32x32
438,metal121.jpg,metal,48x32
466,metal2.jpg,metal,32x32
483,metal26.jpg,metal,40x40
